# RE:SOURCE AI | Task 3: First Diagnosis Agent

**What we are building:** A *read-only, tool-using diagnosis agent* for the broken Random Forest experiment from Task 2. It reads the actual traceback, inspects the broken code, checks the Scikit-learn API, and writes an evidence-backed diagnosis.

**Research safeguards:** The agent must not edit `broken.py`, `experiment_config.json`, `original_metrics.json`, or `split_indices.npz`. It must not claim Gap 1 (reproduced results) or Gap 3 (preserved methodology after repair) is solved yet.

> **Scope:** This first prototype uses deterministic tool selection and rules. It is a real working diagnosis workflow, but **not yet an LLM-powered autonomous repair agent**. That is a later milestone. It never executes untrusted uploaded code.

**What you need:** Your Task 2 output ZIP (`RE_SOURCE_AI_Task2_Outputs.zip`), or the companion example ZIP if you have not saved your own. CPU is sufficient.

## Step 1 · Load the broken experiment from Task 2

Upload the ZIP containing `experiments/broken.py`, `logs/broken_stderr.txt`, and `reference/`. The notebook safely copies only the required files into a new workspace. It will never overwrite the original ZIP.

In [1]:
from pathlib import Path
from zipfile import ZipFile
import sys

CWD = Path.cwd()
options = [
    CWD / 'RE_SOURCE_AI_Task2_Outputs.zip',
    CWD / 'RE_SOURCE_AI_Task2_Example_Outputs.zip',
]
archive_path = next((p for p in options if p.is_file()), None)
if archive_path is None and 'google.colab' in sys.modules:
    from google.colab import files
    print('Upload the ZIP generated by Task 2 (or the example ZIP).')
    uploaded = files.upload()
    matched = [name for name in uploaded
               if name.endswith('.zip')]
    if len(matched) != 1:
        raise ValueError('Please upload exactly one Task 2 ZIP.')
    archive_path = CWD / matched[0]
if archive_path is None:
    raise FileNotFoundError('Place the Task 2 ZIP next to this notebook, then rerun Step 1.')

WORKSPACE = CWD / 'RE_SOURCE_AI_Task3_Workspace'
required = [
    'experiments/broken.py',
    'logs/broken_stderr.txt',
    'reference/experiment_config.json',
    'reference/original_metrics.json',
    'reference/split_indices.npz',
]
WORKSPACE.mkdir(parents=True, exist_ok=True)
with ZipFile(archive_path) as archive:
    names = set(archive.namelist())
    missing = [name for name in required if name not in names]
    if missing:
        raise ValueError(f'This does not look like a Task 2 output ZIP. Missing: {missing}')
    for name in required:
        info = archive.getinfo(name)
        if info.file_size > 2_000_000:
            raise ValueError('Unexpectedly large input: ' + name)
        target = WORKSPACE / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(name))
print('Loaded:', archive_path.name)
print('New read-only diagnostic workspace:', WORKSPACE)
print('Required Task 2 inputs:', len(required))

Upload the ZIP generated by Task 2 (or the example ZIP).


Saving RE_SOURCE_AI_Task2_Outputs.zip to RE_SOURCE_AI_Task2_Outputs.zip
Loaded: RE_SOURCE_AI_Task2_Outputs.zip
New read-only diagnostic workspace: /content/RE_SOURCE_AI_Task3_Workspace
Required Task 2 inputs: 5


## Step 2 · Create the diagnostic tools

Our first agent can make these inspections without modifying experimental code:

- **Read the error log:** What actually failed?
- **Inspect the broken source:** Which line introduced the invalid parameter?
- **Inspect Scikit-learn:** What parameter names does the installed model accept?
- **Inspect the reference:** What do the original scientific settings require?
- **Hash the inputs:** Did the diagnosis leave all source and reference files untouched?

The code below provides those tools. It parses Python syntax but does **not** run `broken.py`.

In [2]:
from __future__ import annotations

import ast
import difflib
import hashlib
import inspect
import json
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from sklearn.ensemble import RandomForestClassifier

ANSI_ESCAPE = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
UNEXPECTED_KW = re.compile(r"unexpected keyword argument\s+['\"]([A-Za-z_][A-Za-z_0-9]*)['\"]", re.IGNORECASE)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(64 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def json_dumps(obj: Any) -> str:
    return json.dumps(obj, indent=2, ensure_ascii=False, sort_keys=True)

class DiagnosticTools:
    """Restrict the agent to read-only inspection of named Task 2 files."""

    ALLOWED_INPUTS = (
        'experiments/broken.py',
        'logs/broken_stderr.txt',
        'reference/experiment_config.json',
        'reference/original_metrics.json',
        'reference/split_indices.npz',
    )

    def __init__(self, root: Path):
        self.root = Path(root).resolve()
        missing = [rel for rel in self.ALLOWED_INPUTS if not (self.root / rel).is_file()]
        if missing:
            raise FileNotFoundError(f'Task 2 input files missing: {missing}')
        self.input_hashes_before = {rel: sha256_file(self.root / rel) for rel in self.ALLOWED_INPUTS}

    def read_failure_log(self) -> dict[str, Any]:
        log = ANSI_ESCAPE.sub('', (self.root / 'logs/broken_stderr.txt').read_text(encoding='utf-8'))
        match = UNEXPECTED_KW.search(log)
        lines = log.splitlines()
        return {
            'source': 'logs/broken_stderr.txt',
            'error_type': 'TypeError' if 'TypeError' in log else ('Unknown' if log.strip() else 'No error recorded'),
            'unexpected_parameter': match.group(1) if match else None,
            'observed_excerpt': '\n'.join(lines[-5:])[-1800:],
        }

    def inspect_broken_source(self, search_term: str | None) -> dict[str, Any]:
        path = self.root / 'experiments/broken.py'
        code = path.read_text(encoding='utf-8')
        ast.parse(code)  # syntactic inspection only; do not execute broken.py
        lines = code.splitlines()
        selected = [
            {'line_number': i, 'text': line.strip()}
            for i, line in enumerate(lines, start=1)
            if search_term and search_term in line
        ]
        # Inspect assignments to params[...] to identify an injected key rename.
        renames = []
        tree = ast.parse(code)
        for node in ast.walk(tree):
            if not isinstance(node, ast.Assign) or len(node.targets) != 1:
                continue
            target, value = node.targets[0], node.value
            if not (isinstance(target, ast.Subscript) and isinstance(target.value, ast.Name)
                    and target.value.id == 'params' and isinstance(target.slice, ast.Constant)
                    and isinstance(target.slice.value, str)):
                continue
            if not (isinstance(value, ast.Call) and isinstance(value.func, ast.Attribute)
                    and isinstance(value.func.value, ast.Name) and value.func.value.id == 'params'
                    and value.func.attr == 'pop' and len(value.args) == 1
                    and isinstance(value.args[0], ast.Constant) and isinstance(value.args[0].value, str)):
                continue
            renames.append({
                'line_number': node.lineno,
                'new_key': target.slice.value,
                'removed_key': value.args[0].value,
                'code': lines[node.lineno - 1].strip(),
            })
        return {
            'source': 'experiments/broken.py',
            'line_evidence': selected[:8],
            'parameter_key_renames': renames,
            'source_was_executed': False,
        }

    def inspect_model_api(self) -> dict[str, Any]:
        sig = inspect.signature(RandomForestClassifier)
        accepted = [k for k in sig.parameters if k != 'self']
        return {
            'tool': 'Python inspect.signature',
            'model': 'sklearn.ensemble.RandomForestClassifier',
            'accepted_parameter_names': accepted,
            'signature': str(sig),
        }

    def inspect_reference_settings(self) -> dict[str, Any]:
        config = json.loads((self.root / 'reference/experiment_config.json').read_text(encoding='utf-8'))
        return {
            'source': 'reference/experiment_config.json',
            'experiment_id': config.get('experiment_id'),
            'dataset': config.get('dataset'),
            'model': config.get('model'),
            'split': config.get('split'),
            'preprocessing': config.get('preprocessing'),
            'evaluation': config.get('evaluation'),
        }

    def inspect_input_integrity(self) -> dict[str, Any]:
        after = {rel: sha256_file(self.root / rel) for rel in self.ALLOWED_INPUTS}
        return {
            'all_inputs_unchanged': self.input_hashes_before == after,
            'modified_inputs': [rel for rel in self.ALLOWED_INPUTS if self.input_hashes_before[rel] != after[rel]],
            'sha256_before': self.input_hashes_before,
            'sha256_after': after,
        }

## Step 3 · Create the agent's decision process

This is what makes the prototype more than a printed explanation. The agent first reads the evidence, **decides which tools to call next**, tests whether candidate explanations agree with the code, reference configuration, and model API, and refuses to guess when the evidence is inadequate.

It suggests a repair for human review, but **does not apply the repair**.

In [3]:
class DiagnosisAgent:
    """Bounded agent: read error, select tools, form evidence-backed diagnosis."""

    def __init__(self, root: Path):
        self.tools = DiagnosticTools(root)
        self.trace: list[dict[str, Any]] = []

    def call(self, name: str, reason: str, **kwargs: Any) -> dict[str, Any]:
        fn = getattr(self.tools, name)
        result = fn(**kwargs)
        self.trace.append({'step': len(self.trace) + 1, 'tool': name, 'reason': reason, 'ok': True})
        return result

    def run(self) -> dict[str, Any]:
        failure = self.call('read_failure_log', 'Read the observed failure before making a hypothesis.')
        bad_key = failure['unexpected_parameter']
        reference = self.call('inspect_reference_settings', 'Read protected scientific requirements; never modify them.')

        if bad_key:
            source = self.call('inspect_broken_source', 'Find where the rejected key entered the source.', search_term=bad_key)
            api = self.call('inspect_model_api', 'Check accepted constructor parameters independently of the error message.')
            accepted = set(api['accepted_parameter_names'])
            expected = set(reference.get('model', {}).get('parameters', {}))
            matches = difflib.get_close_matches(bad_key, sorted(accepted & expected), n=3, cutoff=0.62)
            alternative = matches[0] if matches else None
            rename = next((r for r in source['parameter_key_renames']
                           if r['new_key'] == bad_key and r['removed_key'] == alternative), None)
            if alternative and bad_key not in accepted and rename:
                status = 'DIAGNOSED_REVIEW_REQUIRED'
                diagnosis = f'The script renames valid model parameter {alternative!r} to invalid {bad_key!r} before constructing RandomForestClassifier.'
                proposal = f'Remove the injected parameter-rename statement on line {rename["line_number"]}; retain the reference value for {alternative!r} and all other settings.'
                evidence = [
                    {'source': failure['source'], 'detail': f'Observed TypeError rejects {bad_key!r}.'},
                    {'source': source['source'], 'line': rename['line_number'], 'detail': rename['code']},
                    {'source': reference['source'], 'detail': f'Protected configuration expects {alternative!r}.'},
                    {'source': api['tool'], 'detail': f'RandomForestClassifier accepts {alternative!r} but not {bad_key!r}.'},
                ]
            else:
                status = 'INSUFFICIENT_EVIDENCE'
                diagnosis = 'An unexpected model parameter was reported, but the source/reference/API evidence did not establish a safe single correction.'
                proposal = 'Escalate for manual review; do not modify model settings.'
                evidence = [{'source': failure['source'], 'detail': failure['observed_excerpt']}]
        else:
            source = self.call('inspect_broken_source', 'Read the script without executing it.', search_term=None)
            api = None
            status = 'INSUFFICIENT_EVIDENCE'
            diagnosis = 'The saved error log does not identify an unexpected keyword argument; this diagnostic prototype cannot establish a specific cause.'
            proposal = 'Inspect the full traceback or add another diagnostic tool; do not guess or change methodology.'
            evidence = [{'source': failure['source'], 'detail': failure['observed_excerpt']}]

        integrity = self.call('inspect_input_integrity', 'Prove the source and protected reference inputs were not edited.')
        if not integrity['all_inputs_unchanged']:
            status = 'INTEGRITY_CHECK_FAILED'

        report = {
            'project': 'RE:SOURCE AI',
            'task': 'Task 3 — read-only diagnosis',
            'generated_at_utc': datetime.now(timezone.utc).isoformat(),
            'experiment_id': reference.get('experiment_id'),
            'agent_type': 'deterministic_tool_using_prototype_no_llm',
            'status': status,
            'observed_failure': failure,
            'root_cause_hypothesis': diagnosis,
            'suggested_repair_for_later_review': proposal,
            'evidence': evidence,
            'source_inspection': source,
            'reference_settings': reference,
            'model_api': api,
            'tool_trace': self.trace,
            'integrity': integrity,
            'repair_applied': False,
            'execution_after_repair': 'NOT_TESTED',
            'methodology_preservation': 'NOT_TESTED_NO_REPAIR',
            'result_reproduction': 'NOT_TESTED_NO_REPAIR',
            'research_gap_1': 'Not evaluated yet: require a tested repair and result comparison.',
            'research_gap_3': 'Not evaluated yet: unchanged inputs only prove diagnosis did not alter methodology.',
        }
        out = self.tools.root / 'reports' / 'task3_diagnosis.json'
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(json_dumps(report) + '\n', encoding='utf-8')
        return report

def run_diagnosis(root: str | Path) -> dict[str, Any]:
    return DiagnosisAgent(Path(root)).run()

## Step 4 · Let the Diagnosis Agent investigate

Run the actual agent against the Task 2 files and observe its evidence-backed finding. The result is saved as `reports/task3_diagnosis.json`.

In [4]:
report = run_diagnosis(WORKSPACE)
print('STATUS:', report['status'])
print('OBSERVED ERROR:', report['observed_failure']['observed_excerpt'].splitlines()[-1])
print('ROOT CAUSE:', report['root_cause_hypothesis'])
print('SUGGESTED NEXT ACTION:', report['suggested_repair_for_later_review'])
print('INPUT FILES UNCHANGED:', report['integrity']['all_inputs_unchanged'])
print('REPAIR APPLIED:', report['repair_applied'])
print('\nTOOLS THE AGENT CHOSE:')
for event in report['tool_trace']:
    print(f"  {event['step']}. {event['tool']} | {event['reason']}")

STATUS: DIAGNOSED_REVIEW_REQUIRED
OBSERVED ERROR: TypeError: RandomForestClassifier.__init__() got an unexpected keyword argument 'n_estimator'. Did you mean 'n_estimators'?
ROOT CAUSE: The script renames valid model parameter 'n_estimators' to invalid 'n_estimator' before constructing RandomForestClassifier.
SUGGESTED NEXT ACTION: Remove the injected parameter-rename statement on line 19; retain the reference value for 'n_estimators' and all other settings.
INPUT FILES UNCHANGED: True
REPAIR APPLIED: False

TOOLS THE AGENT CHOSE:
  1. read_failure_log | Read the observed failure before making a hypothesis.
  2. inspect_reference_settings | Read protected scientific requirements; never modify them.
  3. inspect_broken_source | Find where the rejected key entered the source.
  4. inspect_model_api | Check accepted constructor parameters independently of the error message.
  5. inspect_input_integrity | Prove the source and protected reference inputs were not edited.


## Step 5 · Inspect the proof and preserve our research boundaries

The diagnosis should identify the rejected `n_estimator` parameter and explain why `n_estimators` is the valid scientific setting. The input-file hashes should be unchanged.

**Research Gap 1:** Not evaluated yet, because we have not repaired or rerun the experiment.

**Research Gap 3:** Not evaluated yet, because we have not compared a repaired experiment to its original methodology. The current hash check only shows that diagnosis did not change our reference files.

In [5]:
print('EVIDENCE USED:')
for item in report['evidence']:
    line = f" (line {item['line']})" if 'line' in item else ''
    print(f"- {item['source']}{line}: {item['detail']}")

assert report['integrity']['all_inputs_unchanged'], 'The diagnostic tool modified an input!'
assert report['repair_applied'] is False, 'Task 3 must not change the experiment.'
assert report['methodology_preservation'] == 'NOT_TESTED_NO_REPAIR'
assert report['result_reproduction'] == 'NOT_TESTED_NO_REPAIR'
print('\nSafeguards passed: inputs unchanged, no repair, no false reproducibility claim.')
print('Structured report:', WORKSPACE / 'reports' / 'task3_diagnosis.json')

EVIDENCE USED:
- logs/broken_stderr.txt: Observed TypeError rejects 'n_estimator'.
- experiments/broken.py (line 19): params["n_estimator"] = params.pop("n_estimators")
- reference/experiment_config.json: Protected configuration expects 'n_estimators'.
- Python inspect.signature: RandomForestClassifier accepts 'n_estimators' but not 'n_estimator'.

Safeguards passed: inputs unchanged, no repair, no false reproducibility claim.
Structured report: /content/RE_SOURCE_AI_Task3_Workspace/reports/task3_diagnosis.json


## Step 6 · Save the structured report

The JSON file records tool calls, observed errors, sources for every conclusion, recommended repair, protected-file hashes, and the status of our two research gaps. Download the ZIP and keep it for Task 4.

In [6]:
from zipfile import ZipFile, ZIP_DEFLATED
output_zip = CWD / 'RE_SOURCE_AI_Task3_Outputs.zip'
with ZipFile(output_zip, 'w', ZIP_DEFLATED) as archive:
    for path in sorted(WORKSPACE.rglob('*')):
        if path.is_file():
            archive.write(path, arcname=str(path.relative_to(WORKSPACE)))
print('Saved:', output_zip)
print('In Colab, open the Files sidebar to download it.')
# Optional in Colab: from google.colab import files; files.download(str(output_zip))

Saved: /content/RE_SOURCE_AI_Task3_Outputs.zip
In Colab, open the Files sidebar to download it.


## Step 7 · Understand today's outcome

**Completed:** Our first diagnostic workflow autonomously inspects a saved failure, searches the relevant source, checks accepted API parameters, consults protected scientific settings, and produces a structured report without editing research inputs.

**Not completed:** LLM-based reasoning, automated patch generation, running a repaired experiment, and verified scientific recovery.

**Next task (Task 4):** Add a constrained Repair Agent that proposes an editable patch, validates it without changing protected files, and executes only this trusted, controlled experiment. Later we compare repaired metrics and scientific settings to test Gaps

